In [1]:
import glob
import pandas as pd
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import tqdm.auto as tqdm

/home/jburton/nobackup_home/miniforge/envs/plip/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = CLIPModel.from_pretrained("vinid/plip")
processor = CLIPProcessor.from_pretrained("vinid/plip")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [3]:
labels = snakemake.params.labels

In [4]:
image_files = glob.glob(snakemake.input.data_root + "*.jpg")
assert(len(image_files) >= 1) # There must be at least one image to classify.

In [5]:
results = []
results.append(("", *labels))

dataset_name = "_".join(image_files[0].split("/")[1:-1])
for file in tqdm.tqdm(image_files):
    barcode = file.split("/")[-1][:-4] # assumes .jpg.

    image = Image.open(file)
    inputs = processor(text=labels,
                   images=image, return_tensors="pt", padding=True)
    outputs = model(**inputs)
    logits_per_image = outputs.logits_per_image  # this is the image-text similarity score
    probs = logits_per_image.softmax(dim=1).detach().numpy()[0]
    
    results.append((barcode, *probs))

100%|██████████| 4361/4361 [10:47<00:00,  6.74it/s]


In [9]:
pd.DataFrame(results).to_csv(snakemake.output.csv, index=False, header=False)